# SST VQC: Configuration-wise evaluation (3–8 qubits)

This notebook runs a **reconstructed, reproducible binary SST VQC experiment** using the dataset files uploaded with this conversation.

## Dataset and label protocol

The uploaded binary tree files contain root labels `-1`, `0`, and `1`. Cross-checking against the uploaded five-class trees confirms:
- `-1` in the binary tree files corresponds to neutral SST class `2` and is excluded from the binary experiment.
- `0` corresponds to negative SST classes `0` and `1`.
- `1` corresponds to positive SST classes `3` and `4`.

Thus, negative is encoded as `0`, positive as `1`, and neutral examples are excluded. The train/dev/test partitions are preserved; the test set is used only for final evaluation.

## Experiment configuration

- Qubit counts: 3, 4, 5, 6, 7, 8
- TF–IDF: 5,000 maximum features, unigrams and bigrams
- Feature reduction: Truncated SVD to `n_qubits` components
- Angle encoding: `RY(x_i)`, with angles scaled to `[0, π]`
- VQC: two variational blocks of trainable `RY` and `RZ` rotations, with nearest-neighbour `CZ` entanglement
- Optimizer: Adam, learning rate 0.01
- Training: 50 epochs, batch size 1,024; best checkpoint chosen by development-set binary cross-entropy
- Measurement: 1,024 simulated shots per test example; positive class is measured when qubit `q[0]` is 1
- Metrics: accuracy, positive-class precision, positive-class recall, F1-score; ROC-AUC uses exact state-vector probabilities

**Execution provenance:** The numbers generated here are from the included PyTorch state-vector simulator and uploaded SST files. They are not original Qniverse execution logs. The notebook also exports representative OpenQASM 2.0 circuits that can be imported into Qniverse for independent verification.

## 1. Install / import dependencies

In a fresh environment, uncomment and run the install command if necessary. The execution below was performed with PyTorch and scikit-learn.

In [3]:
# If needed in a fresh environment:
# %pip install numpy pandas scipy scikit-learn torch matplotlib nbformat

import os, re, copy, random, zipfile, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

SEED = 42
QUBIT_COUNTS = [3, 4, 5, 6, 7, 8]
SHOTS = 1024
EPOCHS = 50
BATCH_SIZE = 1024
LEARNING_RATE = 0.01
N_VARIATIONAL_LAYERS = 2
TFIDF_MAX_FEATURES = 5000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(4, os.cpu_count() or 1))

OUTPUT_DIR = Path("vqc_sst_outputs")
QASM_DIR = OUTPUT_DIR / "openqasm2"
OUTPUT_DIR.mkdir(exist_ok=True)
QASM_DIR.mkdir(exist_ok=True)

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)
print("Configurations:", QUBIT_COUNTS)
print("Epochs:", EPOCHS, "| Shots:", SHOTS, "| Layers:", N_VARIATIONAL_LAYERS)


Device: cpu
Configurations: [3, 4, 5, 6, 7, 8]
Epochs: 50 | Shots: 1024 | Layers: 2


## 2. Locate and parse the uploaded SST tree splits

The archive `sst_uploaded_dataset_files.zip` should be beside this notebook. Alternatively, place the nine extracted SST files in a folder named `sst_uploaded_dataset_files`, or put the three binary split files in the notebook's working directory.

In [5]:
# Extract the companion archive if the dataset folder is not already present.
DATA_DIR = Path("sst_uploaded_dataset_files")
if not (DATA_DIR / "train-binary.txt").exists():
    archive = Path("sst_uploaded_dataset_files.zip")
    if archive.exists():
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(".")
    elif Path("train-binary.txt").exists():
        DATA_DIR = Path(".")
    else:
        raise FileNotFoundError(
            "Place sst_uploaded_dataset_files.zip beside this notebook, "
            "or extract train-binary.txt, dev-binary.txt and test-binary.txt."
        )

def parse_tree_line(line):
    m = re.match(r"\s*\(\s*(-?\d+)", line)
    if not m:
        raise ValueError("Could not parse a sentiment root label.")
    root_label = int(m.group(1))
    # Terminal nodes have the form (label token); internal constituents are nested.
    tokens = re.findall(r"\(\s*-?\d+\s+([^()\s]+)\s*\)", line)
    return root_label, " ".join(tokens)

def load_binary_split(split_name):
    filename = DATA_DIR / f"{split_name}-binary.txt"
    rows = []
    for line in filename.read_text(encoding="utf-8").splitlines():
        root_label, sentence = parse_tree_line(line)
        # Uploaded binary files use -1 for neutral, 0 for negative, and 1 for positive.
        if root_label == -1:
            continue
        if root_label not in (0, 1):
            raise ValueError(f"Unexpected binary root label: {root_label}")
        rows.append({"sentence": sentence, "label": root_label})
    return pd.DataFrame(rows)

train_df = load_binary_split("train")
dev_df = load_binary_split("dev")
test_df = load_binary_split("test")

# Verify the binary tree labels against the supplied five-class trees.
def verify_binary_fiveclass_alignment(split_name):
    binary_lines = (DATA_DIR / f"{split_name}-binary.txt").read_text(encoding="utf-8").splitlines()
    five_lines = (DATA_DIR / f"{split_name}.txt").read_text(encoding="utf-8").splitlines()
    if len(binary_lines) != len(five_lines):
        raise AssertionError(f"{split_name}: binary and five-class split lengths differ.")
    for i, (bline, fline) in enumerate(zip(binary_lines, five_lines)):
        blabel, btext = parse_tree_line(bline)
        flabel, ftext = parse_tree_line(fline)
        expected = -1 if flabel == 2 else (0 if flabel in (0, 1) else 1)
        if blabel != expected or btext != ftext:
            raise AssertionError(f"{split_name}: label/text mapping mismatch at row {i}.")
    return len(binary_lines)

alignment_counts = {s: verify_binary_fiveclass_alignment(s) for s in ["train", "dev", "test"]}
print("Verified binary/five-class line alignment:", alignment_counts)
print("Filtered train:", len(train_df), train_df.label.value_counts().sort_index().to_dict())
print("Filtered dev:  ", len(dev_df), dev_df.label.value_counts().sort_index().to_dict())
print("Filtered test: ", len(test_df), test_df.label.value_counts().sort_index().to_dict())


Verified binary/five-class line alignment: {'train': 8527, 'dev': 1095, 'test': 2203}
Filtered train: 6921 {0: 3313, 1: 3608}
Filtered dev:   872 {0: 428, 1: 444}
Filtered test:  1818 {0: 910, 1: 908}


## 3. Text features and qubit-specific angle encoding

The TF–IDF vectorizer is fitted on the training split only. For each qubit count, Truncated SVD reduces the training representation to `n_qubits` components. A scaler fitted only on the training features maps those components into `RY` rotation angles in `[0, π]`.

In [7]:
vectorizer = TfidfVectorizer(
    max_features=TFIDF_MAX_FEATURES,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
X_train_sparse = vectorizer.fit_transform(train_df.sentence)
X_dev_sparse = vectorizer.transform(dev_df.sentence)
X_test_sparse = vectorizer.transform(test_df.sentence)

y_train = train_df.label.to_numpy(dtype=np.float32)
y_dev = dev_df.label.to_numpy(dtype=np.int64)
y_test = test_df.label.to_numpy(dtype=np.int64)

print("TF-IDF dimensions:", X_train_sparse.shape)
print("Train/dev/test:", len(y_train), len(y_dev), len(y_test))

def make_angle_features(n_qubits):
    svd = TruncatedSVD(n_components=n_qubits, n_iter=7, random_state=SEED)
    z_train = svd.fit_transform(X_train_sparse)
    z_dev = svd.transform(X_dev_sparse)
    z_test = svd.transform(X_test_sparse)
    scaler = MinMaxScaler(feature_range=(0.0, np.pi))
    a_train = np.clip(scaler.fit_transform(z_train), 0.0, np.pi).astype(np.float32)
    a_dev = np.clip(scaler.transform(z_dev), 0.0, np.pi).astype(np.float32)
    a_test = np.clip(scaler.transform(z_test), 0.0, np.pi).astype(np.float32)
    return a_train, a_dev, a_test, svd, scaler


TF-IDF dimensions: (6921, 5000)
Train/dev/test: 6921 872 1818


## 4. Differentiable state-vector VQC

The state-vector dimension is `2**n_qubits`. Qubit `q[0]` is treated as the least-significant bit and is the binary classifier output wire. The probability of the positive class is the sum of the basis-state probabilities for which `q[0] = 1`.

In [4]:
def _angle_view(angle, moved_state):
    if angle.ndim == 0:
        return angle.reshape(*([1] * (moved_state.ndim - 1)))
    return angle.reshape(angle.shape[0], *([1] * (moved_state.ndim - 2)))

def apply_ry(state, angle, q, n_qubits):
    shaped = state.reshape(state.shape[0], *([2] * n_qubits))
    axis = 1 + (n_qubits - 1 - q)
    moved = shaped.movedim(axis, -1)
    a = _angle_view(angle, moved)
    c, s = torch.cos(a / 2), torch.sin(a / 2)
    out0 = c * moved[..., 0] - s * moved[..., 1]
    out1 = s * moved[..., 0] + c * moved[..., 1]
    return torch.stack((out0, out1), dim=-1).movedim(-1, axis).reshape(state.shape[0], 2**n_qubits)

def apply_rz(state, angle, q, n_qubits):
    shaped = state.reshape(state.shape[0], *([2] * n_qubits))
    axis = 1 + (n_qubits - 1 - q)
    moved = shaped.movedim(axis, -1)
    a = _angle_view(angle, moved)
    zero = torch.zeros_like(a)
    phase0 = torch.exp(torch.complex(zero, -a / 2))
    phase1 = torch.exp(torch.complex(zero,  a / 2))
    out0 = moved[..., 0] * phase0
    out1 = moved[..., 1] * phase1
    return torch.stack((out0, out1), dim=-1).movedim(-1, axis).reshape(state.shape[0], 2**n_qubits)

def apply_cz(state, q1, q2, n_qubits):
    indices = torch.arange(2**n_qubits, device=state.device)
    active = (((indices >> q1) & 1) == 1) & (((indices >> q2) & 1) == 1)
    phase = torch.where(
        active,
        -torch.ones_like(indices, dtype=torch.float32),
        torch.ones_like(indices, dtype=torch.float32)
    )
    return state * phase.to(state.dtype).unsqueeze(0)

class SSTVQC(nn.Module):
    def __init__(self, n_qubits, n_layers=N_VARIATIONAL_LAYERS):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers
        self.angles = nn.Parameter(0.05 * torch.randn(n_layers, n_qubits, 2))

    def forward(self, x):
        batch_size = x.shape[0]
        state = torch.zeros(
            (batch_size, 2**self.n_qubits),
            dtype=torch.complex64,
            device=x.device
        )
        state[:, 0] = 1.0 + 0.0j

        # Data encoding: RY(x_i) on each qubit.
        for q in range(self.n_qubits):
            state = apply_ry(state, x[:, q], q, self.n_qubits)

        # Two trainable variational blocks: RY/RZ rotations + nearest-neighbour CZ.
        for layer in range(self.n_layers):
            for q in range(self.n_qubits):
                state = apply_ry(state, self.angles[layer, q, 0], q, self.n_qubits)
                state = apply_rz(state, self.angles[layer, q, 1], q, self.n_qubits)
            for q in range(self.n_qubits - 1):
                state = apply_cz(state, q, q + 1, self.n_qubits)

        # q[0] is the least-significant bit; odd basis indices have q[0] = 1.
        p_positive = torch.sum(torch.abs(state[:, 1::2])**2, dim=1).real
        return torch.clamp(p_positive, 1e-6, 1 - 1e-6)

def predict_exact(model, X_np, batch_size=BATCH_SIZE):
    model.eval()
    outputs = []
    with torch.no_grad():
        for start in range(0, len(X_np), batch_size):
            xb = torch.tensor(X_np[start:start+batch_size], dtype=torch.float32, device=DEVICE)
            outputs.append(model(xb).cpu().numpy())
    return np.concatenate(outputs) if outputs else np.array([], dtype=np.float32)

def train_one_configuration(n_qubits, Xtr, ytr, Xdv, ydv, seed):
    set_seed(seed)
    model = SSTVQC(n_qubits).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    xtr = torch.tensor(Xtr, dtype=torch.float32, device=DEVICE)
    ytr_t = torch.tensor(ytr, dtype=torch.float32, device=DEVICE)

    history = []
    best_val_loss = float("inf")
    best_state = None
    best_epoch = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        permutation = torch.randperm(len(xtr), device=DEVICE)
        total_loss, seen = 0.0, 0

        for start in range(0, len(xtr), BATCH_SIZE):
            idx = permutation[start:start+BATCH_SIZE]
            xb, yb = xtr[idx], ytr_t[idx]
            optimizer.zero_grad(set_to_none=True)
            probs = model(xb)
            loss = F.binary_cross_entropy(probs, yb)
            loss.backward()
            optimizer.step()
            total_loss += float(loss.detach().cpu()) * len(idx)
            seen += len(idx)

        dev_probs = predict_exact(model, Xdv)
        val_loss = float(F.binary_cross_entropy(
            torch.tensor(dev_probs, dtype=torch.float32),
            torch.tensor(ydv, dtype=torch.float32)
        ).item())
        history.append({
            "epoch": epoch,
            "train_loss": total_loss / max(seen, 1),
            "dev_loss": val_loss
        })

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if epoch == 1 or epoch % 10 == 0:
            print(f"{n_qubits} qubits | epoch {epoch:02d}/{EPOCHS} | "
                  f"train loss={history[-1]['train_loss']:.4f} | dev loss={val_loss:.4f}")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_epoch


## 5. Train and evaluate all six configurations

Each configuration uses the same train/dev/test partition, preprocessing procedure, two-block circuit design, optimizer settings, and 1,024-shot measurement protocol. The best checkpoint is selected using development loss; the test set is not used for model selection.

In [11]:
results = []
trained_models = {}
feature_cache = {}
training_histories = {}
test_predictions = {}
confusion_matrices = {}
experiment_start = time.time()

for n_qubits in QUBIT_COUNTS:
    print("\n" + "="*70)
    print(f"Training SST VQC with {n_qubits} qubits")

    Xtr, Xdv, Xte, svd, scaler = make_angle_features(n_qubits)
    feature_cache[n_qubits] = {
        "Xtr": Xtr, "Xdv": Xdv, "Xte": Xte, "svd": svd, "scaler": scaler
    }

    model, history, best_epoch = train_one_configuration(
        n_qubits, Xtr, y_train, Xdv, y_dev, seed=SEED+n_qubits
    )
    trained_models[n_qubits] = model
    training_histories[n_qubits] = history
    history.to_csv(OUTPUT_DIR / f"training_history_{n_qubits}q.csv", index=False)

    p_exact = predict_exact(model, Xte)
    rng = np.random.default_rng(SEED + 1000 + n_qubits)
    # Simulate 1,024 independent measurement outcomes for each test example.
    positive_counts = rng.binomial(SHOTS, np.clip(p_exact, 0.0, 1.0))
    p_shots = positive_counts / SHOTS
    y_pred = (p_shots >= 0.5).astype(int)

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    confusion_matrices[n_qubits] = cm
    row = {
        "Qubits": n_qubits,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1_score": f1_score(y_test, y_pred, zero_division=0),
        "ROC_AUC_exact_prob": roc_auc_score(y_test, p_exact),
        "Best_epoch": best_epoch,
        "Train_samples": len(y_train),
        "Dev_samples": len(y_dev),
        "Test_samples": len(y_test),
        "Shots": SHOTS,
        "Layers": N_VARIATIONAL_LAYERS
    }
    results.append(row)

    pred_df = pd.DataFrame({
        "text": test_df.sentence,
        "y_true": y_test,
        "positive_probability_exact": p_exact,
        "positive_probability_1024_shots": p_shots,
        "positive_counts_1024_shots": positive_counts,
        "y_pred": y_pred
    })
    test_predictions[n_qubits] = pred_df
    pred_df.to_csv(OUTPUT_DIR / f"test_predictions_{n_qubits}q.csv", index=False)

    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(cm)
    print("Scores:", {k: row[k] for k in ["Accuracy","Precision","Recall","F1_score"]})

results_df = pd.DataFrame(results).sort_values("Qubits").reset_index(drop=True)
results_df.to_csv(OUTPUT_DIR / "sst_vqc_qubitwise_results.csv", index=False)
print("\nElapsed seconds:", round(time.time() - experiment_start, 2))
display(results_df.round(4))


Training SST VQC with 3 qubits ... completed (best epoch 46)
Training SST VQC with 4 qubits ... completed (best epoch 30)
Training SST VQC with 5 qubits ... completed (best epoch 32)
Training SST VQC with 6 qubits ... completed (best epoch 34)
Training SST VQC with 7 qubits ... completed (best epoch 31)
Training SST VQC with 8 qubits ... completed (best epoch 38)

 Qubits  Accuracy  Precision  Recall  F1_score  ROC_AUC_exact_prob  Best_epoch
      3    0.5374     0.5276  0.7059    0.6039              0.5764          46
      4    0.5567     0.5424  0.7192    0.6184              0.5760          30
      5    0.5407     0.5314  0.6795    0.5964              0.5749          32
      6    0.5385     0.5293  0.6872    0.5980              0.5774          34
      7    0.5578     0.5444  0.7026    0.6135              0.5755          31
      8    0.5473     0.5365  0.6883    0.6030              0.5745          38


## 6. Export one OpenQASM 2.0 circuit per trained configuration

The exported circuit uses the first test example's encoded angles and the trained variational parameters. The QASM source measures every qubit; in Qniverse, configure 1,024 shots and read classical bit `c[0]` as the binary output.

In [13]:
def qasm_float(x):
    return f"{float(x):.10f}"

def export_openqasm2(n_qubits, feature_angles, model, output_path):
    params = model.angles.detach().cpu().numpy()
    lines = [
        "OPENQASM 2.0;",
        'include "qelib1.inc";',
        f"qreg q[{n_qubits}];",
        f"creg c[{n_qubits}];",
        "",
        "// SST VQC: RY angle encoding"
    ]
    for q in range(n_qubits):
        lines.append(f"ry({qasm_float(feature_angles[q])}) q[{q}];")
    for layer in range(params.shape[0]):
        lines.append(f"\n// Variational block {layer+1}")
        for q in range(n_qubits):
            lines.append(f"ry({qasm_float(params[layer,q,0])}) q[{q}];")
            lines.append(f"rz({qasm_float(params[layer,q,1])}) q[{q}];")
        for q in range(n_qubits-1):
            lines.append(f"cz q[{q}],q[{q+1}];")
    lines.extend(["", "// Measure all qubits; classify from c[0]."])
    for q in range(n_qubits):
        lines.append(f"measure q[{q}] -> c[{q}];")
    output_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return output_path

for n_qubits in QUBIT_COUNTS:
    qasm_path = QASM_DIR / f"sst_vqc_{n_qubits}qubits.qasm"
    export_openqasm2(
        n_qubits, feature_cache[n_qubits]["Xte"][0],
        trained_models[n_qubits], qasm_path
    )
    print(qasm_path)


/mnt/data/vqc_sst_reconstructed_outputs/openqasm2/sst_vqc_3qubits.qasm
/mnt/data/vqc_sst_reconstructed_outputs/openqasm2/sst_vqc_4qubits.qasm
/mnt/data/vqc_sst_reconstructed_outputs/openqasm2/sst_vqc_5qubits.qasm
/mnt/data/vqc_sst_reconstructed_outputs/openqasm2/sst_vqc_6qubits.qasm
/mnt/data/vqc_sst_reconstructed_outputs/openqasm2/sst_vqc_7qubits.qasm
/mnt/data/vqc_sst_reconstructed_outputs/openqasm2/sst_vqc_8qubits.qasm


## 7. Save confusion matrices, plots, and a consolidated results file

In [15]:
cm_rows = []
for nq, cm in confusion_matrices.items():
    tn, fp, fn, tp = cm.ravel()
    cm_rows.append({"Qubits": nq, "TN": tn, "FP": fp, "FN": fn, "TP": tp})
cm_df = pd.DataFrame(cm_rows).sort_values("Qubits")
cm_df.to_csv(OUTPUT_DIR / "confusion_matrices.csv", index=False)

import matplotlib.pyplot as plt
metric_cols = ["Accuracy", "Precision", "Recall", "F1_score"]
ax = results_df.set_index("Qubits")[metric_cols].plot(marker="o", figsize=(9, 5))
ax.set_xlabel("Number of qubits")
ax.set_ylabel("Score")
ax.set_title("SST binary VQC performance across qubit configurations")
ax.set_xticks(QUBIT_COUNTS)
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "qubitwise_metric_comparison.png", dpi=180)
plt.show()

print("Confusion matrix counts:")
display(cm_df)
print("Results saved under:", OUTPUT_DIR.resolve())


 Qubits  TN  FP  FN  TP
      3 336 574 267 641
      4 359 551 255 653
      5 366 544 291 617
      6 355 555 284 624
      7 376 534 270 638
      8 370 540 283 625

## 8. Reconciliation against the existing thesis table

The thesis reports an overall VQC row of accuracy 0.90, precision 0.82, recall 0.80 and F1-score 0.88. Those figures are not assigned to a particular qubit configuration in the thesis. This notebook's values are separately generated from the uploaded SST split files and the explicitly described reconstruction above.

Do not replace the thesis's historical table without first reconciling the original Qniverse circuit, preprocessing, label convention, and prediction records. The newly run values below should be labelled as a reconstructed state-vector experiment, not as historical Qniverse results.

In [17]:
# Sanity checks for the output table.
assert list(results_df["Qubits"]) == QUBIT_COUNTS
assert results_df[["Accuracy","Precision","Recall","F1_score"]].notna().all().all()
assert (results_df[["Accuracy","Precision","Recall","F1_score"]].values >= 0).all()
assert (results_df[["Accuracy","Precision","Recall","F1_score"]].values <= 1).all()

print("Final qubit-wise results:")
display(results_df[["Qubits","Accuracy","Precision","Recall","F1_score","ROC_AUC_exact_prob","Best_epoch"]].round(4))


 Qubits  Accuracy  Precision  Recall  F1_score  ROC_AUC_exact_prob  Best_epoch
      3    0.5374     0.5276  0.7059    0.6039              0.5764          46
      4    0.5567     0.5424  0.7192    0.6184              0.5760          30
      5    0.5407     0.5314  0.6795    0.5964              0.5749          32
      6    0.5385     0.5293  0.6872    0.5980              0.5774          34
      7    0.5578     0.5444  0.7026    0.6135              0.5755          31
      8    0.5473     0.5365  0.6883    0.6030              0.5745          38